# FIELDCARD – TRL 4 retrospective validation

This notebook runs the pre-registered test in `configs/preregistered_trl4.yaml` **once**, on open data around DLG IPZ Bernburg (Saxony-Anhalt, Germany):

1. DWD station weather → FAO-56 ET0 → drought / normal seasons from station rain only
2. Thünen crop-type maps → winter-wheat and maize fields (≥ 5 ha), random sample with a fixed seed
3. Sentinel-2 L2A (Planetary Computer) → per-field NDVI, NDMI, NDRE
4. `fieldcard` → weekly cards → criteria A, B and C

**Rules for this run**
- Every setting comes from the config file in the cloned commit. Do not edit settings in this notebook.
- Run the cells top to bottom. Intermediate results are cached in Google Drive, so a dropped connection resumes where it stopped.
- The results are reported as they come out.

In [1]:
# 0. Get the exact pre-registered commit, install, run the unit tests
!git clone -q https://github.com/seaycon/fieldcard.git /content/fieldcard
%cd /content/fieldcard
!pip install -q -e ".[data,test]"
!python -m pytest -q
import subprocess
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print("commit used for this run:", COMMIT)

/content/fieldcard
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fieldcard (pyproject.toml) ... done
...............................                                          [100%]
31 passed in 7.56s
commit used for this run: ccb84fd4d0533e6f49537a1b39261c7f7b8f55f5


In [2]:
# 0b. Settings, cache folder in Google Drive
import os, json, time, datetime
import yaml, requests
import numpy as np, pandas as pd, geopandas as gpd
from google.colab import drive
drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/fieldcard_trl4"
os.makedirs(CACHE, exist_ok=True)

os.environ.update({"GDAL_HTTP_MAX_RETRY": "5", "GDAL_HTTP_RETRY_DELAY": "3",
                   "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"})
import fieldcard as fc
from fieldcard import sources as src
from fieldcard.evaluation import classify_seasons, trl4_criteria

cfg = yaml.safe_load(open("configs/preregistered_trl4.yaml"))
assert fc.__version__ == cfg["package_version"], "package and config versions differ"
SEASONS = cfg["seasons"]
print("seasons:", SEASONS, "| fieldcard", fc.__version__)

Mounted at /content/drive
seasons: [2017, 2018, 2019, 2020, 2021, 2022] | fieldcard 0.2.3


## 1. Weather and season classes (station rain only)

In [3]:
W, SC = cfg["weather"], cfg["season_classes"]
dwd = src.dwd_daily_kl(W["station_id"])
dwd.to_parquet(f"{CACHE}/dwd_{W['station_id']}.parquet")
print("DWD files:", dwd["source_file"].unique().tolist())

classes = classify_seasons(dwd.set_index("date")["RSK"], SEASONS, months=tuple(SC["months"]),
                           normal_period=tuple(SC["normal_period"]),
                           drought_below=SC["drought_below_ratio"],
                           normal_at_or_above=SC["normal_at_or_above_ratio"])
print("Apr-Aug rain, 1991-2020 mean:", classes.attrs["normal_mean_mm"], "mm from",
      classes.attrs["normal_years_used"], "years")
display(classes)

wx = src.weather_table(dwd, W["lat"], W["lon"], W["elevation_m"], f"{min(SEASONS)}-01-01",
                       f"{max(SEASONS)}-12-31", W["wind_height_m"])
wx.to_parquet(f"{CACHE}/weather.parquet")
print("days filled from ERA5-Land -> ET0:", int(wx["filled_et0"].sum()), "| rain:", int(wx["filled_precip"].sum()))
display(wx.assign(year=wx.date.dt.year).query("date.dt.month >= 4 and date.dt.month <= 8")
          .groupby("year")[["precip", "et0"]].sum().round(0))

DWD files: ['tageswerte_KL_00445_18980101_20251231_hist.zip']
Apr-Aug rain, 1991-2020 mean: 262.9 mm from 30 years


,season,rain_mm,ratio,class
0,2017,279.1,1.061,normal
1,2018,167.7,0.638,drought
2,2019,186.2,0.708,drought
3,2020,230.3,0.876,excluded
4,2021,306.8,1.167,normal
5,2022,140.5,0.534,drought


days filled from ERA5-Land -> ET0: 2191 | rain: 12


,precip,et0
year,,
2017,279.0,499.0
2018,168.0,665.0
2019,186.0,607.0
2020,230.0,581.0
2021,307.0,509.0
2022,140.0,624.0


## 2. Fields from the Thünen crop-type maps

In [4]:
F = cfg["fields"]
sld = requests.get(src.THUENEN_LEGEND, timeout=60).text
prop, legend = src.parse_sld(sld)
prop = F["class_attribute"] or prop
codes = F["class_codes"] or src.crop_codes(legend, F["legend_labels"])
LEGEND_CODES = legend["code"].tolist()
print("class attribute in legend:", prop, "(if missing or absent from a file, it is detected from the values)")
print("codes used:", codes)
display(legend)
assert all(codes.values()), "a crop matched no legend class: stop and report, do not guess"

fields_by_season = {}
for y in SEASONS:
    path = f"{CACHE}/fields_{y}.gpkg"
    if os.path.exists(path):
        f = gpd.read_file(path)
    else:
        t0 = time.time()
        cand = src.read_thuenen_fields(y, cfg["study_area"]["bbox_lonlat"], prop, codes,
                                       F["min_area_ha"], F["inner_buffer_m"], legend_codes=LEGEND_CODES)
        print(y, "candidates:", cand.groupby("crop").size().to_dict(), f"({time.time() - t0:.0f} s)")
        f = src.sample_fields(cand, F["n_per_crop_per_season"], F["seed"])
        f.to_file(path, driver="GPKG")
    fields_by_season[y] = f
    print(y, "sampled:", f.groupby("crop").size().to_dict())

class attribute in legend: ctm_majority (if missing or absent from a file, it is detected from the values)
codes used: {'winter_wheat': ['1101'], 'maize': ['1300']}


,code,label
0,200,Permanent grassland
1,1101,Winter wheat
2,1102,Winter barley
3,1103,Winter rye
4,1201,Spring barley
5,1202,Spring oat
6,1300,Maize
7,1401,Potato
8,1402,Sugar beet
9,1501,Winter rapeseed


2017 sampled: {'maize': 30, 'winter_wheat': 30}
2018 sampled: {'maize': 30, 'winter_wheat': 30}
2019 sampled: {'maize': 30, 'winter_wheat': 30}
2020 sampled: {'maize': 30, 'winter_wheat': 30}
2021 sampled: {'maize': 30, 'winter_wheat': 30}
2022 sampled: {'maize': 30, 'winter_wheat': 30}


## 3. Sentinel-2 per-field indices (one window per acquisition day, cached per season)

In [5]:
S = cfg["sentinel2"]
obs_parts = []
for y in SEASONS:
    path = f"{CACHE}/s2_obs_{y}.parquet"
    if os.path.exists(path):
        o = pd.read_parquet(path)
    else:
        t0 = time.time()
        f = fields_by_season[y]
        bbox = tuple(f.to_crs(4326).total_bounds)
        items = src.s2_search(bbox, f"{y}-{S['period'][0]}", f"{y}-{S['period'][1]}", S["max_scene_cloud_pct"])
        pb = pd.Series([it.properties.get("s2:processing_baseline") for it in items]).value_counts().to_dict()
        print(f"{y}: {len(items)} items, processing baselines {pb}")
        o = src.extract_field_indices(items, f, S["resolution_m"], S["crs"], tuple(S["valid_scl"]),
                                      S["min_valid_fraction"])
        o.to_parquet(path)
        print(f"{y}: {len(o)} field observations on {o['date'].nunique()} days, {(time.time() - t0) / 60:.1f} min")
    obs_parts.append(o)
obs = pd.concat(obs_parts, ignore_index=True)
obs["date"] = pd.to_datetime(obs["date"])
days = obs.groupby(obs["date"].dt.year)["date"].nunique()
KEEP = [y for y in SEASONS if days.get(y, 0) >= cfg["min_scenes_per_season"]]
print("usable days per season:", days.to_dict())
print("seasons kept:", KEEP)

usable days per season: {2017: 29, 2018: 57, 2019: 48, 2020: 57, 2021: 51, 2022: 54}
seasons kept: [2017, 2018, 2019, 2020, 2021, 2022]


## 4. Weekly cards and criteria A, B, C

In [6]:
M, C = cfg["model"], cfg["criteria"]
cls = classes[classes["season"].isin(KEEP)].reset_index(drop=True)
NORMAL = cls.loc[cls["class"].isin(["normal", "normal_fallback"]), "season"].tolist()
DROUGHT = cls.loc[cls["class"] == "drought", "season"].tolist()
print("reference (normal) seasons:", NORMAL, "| drought seasons:", DROUGHT)
assert NORMAL and DROUGHT, "need at least one normal and one drought season: stop and report"

weekly_parts = []
for crop in ("winter_wheat", "maize"):
    pars = {**M["common"], **M[crop]}
    pars["heat_doy"] = tuple(pars["heat_doy"])
    p = fc.Params(**pars)
    o = obs[(obs["crop"] == crop) & obs["date"].dt.year.isin(KEEP)]
    w = fc.run_pipeline(o, wx, p, baseline_years=[y for y in NORMAL], reference_mode=M["reference_mode"])
    weekly_parts.append(w)
weekly = pd.concat(weekly_parts, ignore_index=True)
summary = fc.season_summary(weekly)
res = trl4_criteria(summary, cls, C["flagged_min_weeks"], C["A_drought_separation_pp_min"],
                    C["B_normal_flag_pct_max"], C["C_median_lead_days_min"])
print(json.dumps(res, indent=2, default=str))

reference (normal) seasons: [2017, 2021] | drought seasons: [2018, 2019, 2022]
{
  "fields": {
    "drought_field_seasons": 180,
    "normal_field_seasons": 120,
    "excluded_field_seasons": 60
  },
  "A_drought_separation_pp": 67.5,
  "A_pass": true,
  "pct_flagged_drought": 68.3,
  "B_normal_flag_pct": 0.8,
  "B_pass": true,
  "C_median_lead_days": 0.0,
  "C_pass": false,
  "C_counts": {
    "with_warning_and_loss": 135,
    "left_censored_lower_bound": 0,
    "warning_only": 2,
    "loss_only": 19,
    "negative_leads": 51
  },
  "thresholds": {
    "A_min_pp": 30,
    "B_max_pct": 20,
    "C_min_days": 7,
    "flagged_min_weeks": 2
  },
  "by_season_and_crop": [
    {
      "season": 2017,
      "crop": "maize",
      "fields": 30,
      "pct_flagged": 0.0,
      "median_lead_days": null
    },
    {
      "season": 2017,
      "crop": "winter_wheat",
      "fields": 30,
      "pct_flagged": 0.0,
      "median_lead_days": null
    },
    {
      "season": 2018,
      "crop": "maiz

In [7]:
# 5. Save the evidence (Drive + local) - commit trl4_summary.json to the repository afterwards
out = {
    "protocol": "configs/preregistered_trl4.yaml", "protocol_version": cfg["protocol_version"],
    "commit": COMMIT, "fieldcard_version": fc.__version__,
    "run_utc": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "season_classes": classes.to_dict("records"), "normal_period_mean_mm": classes.attrs["normal_mean_mm"],
    "seasons_kept": KEEP, "usable_days_per_season": {int(k): int(v) for k, v in days.items()},
    "field_seasons": {"total": int(summary["parcel_id"].nunique()),
                      "by_crop": summary.groupby("crop").size().to_dict()},
    "criteria": res,
}
for d in (".", CACHE):
    json.dump(out, open(f"{d}/trl4_summary.json", "w"), indent=2, default=str)
summary.to_csv(f"{CACHE}/trl4_field_seasons.csv", index=False)
weekly.to_parquet(f"{CACHE}/trl4_weekly.parquet")
print("saved to", CACHE)

/tmp/ipykernel_8270/4002730389.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_utc": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",


saved to /content/drive/MyDrive/fieldcard_trl4


In [8]:
# 6. Sentence for Section 2.4 (numbers straight from the run; copy as printed)
used = cls[cls["class"].isin(["drought", "normal", "normal_fallback"])]["season"].tolist()
excluded = [y for y in KEEP if y not in used]
n_fs = res["fields"]["drought_field_seasons"] + res["fields"]["normal_field_seasons"]
txt = (f"The chain was validated retrospectively on {n_fs} maize and winter-wheat field-seasons around "
       f"Bernburg (Saxony-Anhalt) in {len(used)} seasons ({', '.join(map(str, used))}), contrasting the drought "
       f"seasons {', '.join(map(str, DROUGHT))} with the normal seasons {', '.join(map(str, NORMAL))}, with public "
       f"Sentinel-2 data and DWD station weather, against three criteria fixed before the run: (A) in drought "
       f"seasons, the share of fields flagged with water deficit exceeded the normal-season share by "
       f"{res['A_drought_separation_pp']} percentage points (criterion: ≥30); (B) in normal seasons, "
       f"{res['B_normal_flag_pct']}% of fields were flagged (≤20%); (C) the median warning came "
       f"{res['C_median_lead_days']} days before visible greenness loss (≥7).")
print(txt)
if excluded:
    print("\nseasons between the drought and normal thresholds (not used in A-C):", excluded)
print("\nA pass:", res["A_pass"], "| B pass:", res["B_pass"], "| C pass:", res["C_pass"])

The chain was validated retrospectively on 300 maize and winter-wheat field-seasons around Bernburg (Saxony-Anhalt) in 5 seasons (2017, 2018, 2019, 2021, 2022), contrasting the drought seasons 2018, 2019, 2022 with the normal seasons 2017, 2021, with public Sentinel-2 data and DWD station weather, against three criteria fixed before the run: (A) in drought seasons, the share of fields flagged with water deficit exceeded the normal-season share by 67.5 percentage points (criterion: ≥30); (B) in normal seasons, 0.8% of fields were flagged (≤20%); (C) the median warning came 0.0 days before visible greenness loss (≥7).

seasons between the drought and normal thresholds (not used in A-C): [2020]

A pass: True | B pass: True | C pass: False
